# Chapter 10 — Error Handling in Multi-agent Systems · Homework (self-test)

**Goal:** check that you can classify OpenAI failures, keep retries bounded, return errors and partial results as data, and turn worker outcomes into honest coordinator decisions and coverage-annotated reports, without looking at the answers.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 90–120 minutes
- **Estimated API cost:** $0 (every graded part runs offline on a mock HTTP layer or `agents.testing.ScriptedModel`)
- **Prerequisites:** the repo venv from [00-prerequisites](../../00-prerequisites/README.md). No API key is needed.

### How to use this notebook

1. Answer the **quiz** (Part A) in the `answers` cell, then run the grader. Answers are stored as hashes, so you cannot peek.
2. Solve each **coding exercise** (Part B): replace the `TODO` stub, then run its check cell. A check never crashes the notebook; it prints ✓ or a hint.
3. Write your answer to the **architecture scenario** (Part C) and compare it with the rubric.
4. Run the **scorecard**. The pass mark is **72%** (quiz questions + exercises combined), the same bar as the Claude path.
5. Solutions are at the very bottom. Try first.

In [ ]:
import asyncio
import hashlib
import json
import os
import time
from typing import Literal

import httpx2
import openai
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pydantic import BaseModel

from agents import Agent, RunConfig, RunContextWrapper, Runner, function_tool
from agents.testing import ScriptedModel, assistant_message, function_call

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OFFLINE = RunConfig(tracing_disabled=True)
EX_RESULTS: dict[int, bool] = {}
QUIZ_SCORE: dict[str, int] = {"correct": 0, "total": 10}


def record(n: int, passed: bool, hint: str = "") -> None:
    EX_RESULTS[n] = passed
    print(f"✓ Exercise {n} passed" if passed else f"✗ Exercise {n} not passed yet. {hint}")


# --- Given: shared types and helpers (same as in the practice notebook) ---
NON_RETRYABLE_429 = {"credit_balance_exhausted", "organization_spend_limit_exceeded",
                     "project_spend_limit_exceeded", "organization_usage_limit_exceeded"}


class Finding(BaseModel):
    title: str
    source: str


class WorkerResult(BaseModel):
    topic: str
    status: Literal["success", "partial_failure", "failure"]
    failure_type: Literal["timeout", "rate_limited", "validation", "permission",
                          "policy_blocked", "max_turns", "unknown"] | None
    retryable: bool
    attempted_query: str
    findings: list[Finding]
    alternative_approaches: list[str]
    coverage_impact: str | None


class ToolFailure(Exception):
    """Raised by tools for anticipated failures."""

    def __init__(self, category: str, message: str, retryable: bool, suggestion: str | None = None):
        super().__init__(message)
        self.category, self.retryable, self.suggestion = category, retryable, suggestion


def mock_client(handler, **kwargs) -> OpenAI:
    """A real OpenAI client whose HTTP layer is a local function. No network, no key needed."""
    return OpenAI(api_key="sk-offline-demo", http_client=httpx2.Client(transport=httpx2.MockTransport(handler)), **kwargs)


def error_response(status: int, code: str | None = None, type_: str | None = None, param: str | None = None):
    body = {"error": {"message": f"HTTP {status} ({code})", "type": type_, "code": code, "param": param}}
    return httpx2.Response(status, json=body, headers={"x-request-id": f"req_{status}"})


def raise_from(handler) -> openai.APIError:
    """Return the exception the SDK raises for one request (no retries) to a mock server."""
    try:
        mock_client(handler, max_retries=0).responses.create(model=MODEL, input="ping")
    except openai.APIError as exc:
        return exc
    raise AssertionError("the mock server answered successfully")


def read_timeout(request):
    raise httpx2.ReadTimeout("read timed out", request=request)


def connect_failure(request):
    raise httpx2.ConnectError("DNS lookup failed", request=request)


print("Model (not used by graded parts):", MODEL)

## Part A — Quiz (10 questions)

**Q1.** A batch enrichment job sees two kinds of `openai.RateLimitError` overnight: some with `exc.code == "slow_down"`, others with `exc.code == "project_spend_limit_exceeded"`. What is the right handling?

- A) Retry both with exponential backoff; they are the same HTTP status.
- B) Back off and retry `slow_down` (honor `Retry-After`, ramp traffic gradually); treat `project_spend_limit_exceeded` as a non-retryable business error that stops new work and alerts the project owner.
- C) Treat both as permission errors and escalate to security.
- D) Switch both to `background=True` so rate limits no longer apply.

**Q2.** Your request logs show every failed `429` and `503` appearing **three times** with the same body, although your application code calls `responses.create` once and has no retry loop. What explains it, and what do you do if you add your own retry layer?

- A) The API duplicates requests on overload; nothing can be done.
- B) Background mode is enabled by default.
- C) The model streams partial responses; disable streaming.
- D) The `openai` client retries `408`, `409`, `429` and `≥500` twice by default. If your layer owns retries, set `max_retries=0` for those calls (`with_options`) or count the SDK's attempts in your budget.

**Q3.** A coordinator uses `worker.as_tool(...)` for three researchers. When a researcher's model call fails, the coordinator's trace shows the tool output "An error occurred while running the tool. Please try again." and the coordinator calls the same tool four more times. What is the best fix?

- A) Pass a `failure_error_function` to `as_tool` that returns a structured error (category, retryable, suggestion), and a `custom_output_extractor` so successes are explicit `WorkerResult` JSON.
- B) Set `failure_error_function=None` on `as_tool` so the whole coordinator run fails fast.
- C) Raise the coordinator's `max_turns` so it has room for retries.
- D) Replace agents-as-tools with handoffs so the researcher owns the conversation.

**Q4.** A model call gets `503 server_is_overloaded` with the header `Retry-After: 300`. The `openai` client runs on its defaults, yet your logs show a single request and an immediate `InternalServerError`. What explains it, and what should your code do?

- A) The client never retries a `503`; retry it yourself in a tight loop.
- B) `max_retries` defaults to 0 for the Responses API; set it to 2.
- C) The client skips its retries when `Retry-After` is above 120 s and hands the decision to you. The error is still transient: wait only if the delay fits the remaining deadline, otherwise annotate the gap (or let a circuit breaker fail fast).
- D) The client found the header invalid; ignore it and retry immediately.

**Q5.** An agent uses `ModelSettings(retry=ModelRetrySettings(max_retries=3, policy=retry_policies.http_status([429, 503])))`. When prepaid credits run out, every run makes four failing model calls before raising. What is the best change?

- A) Lower `max_retries` to 1.
- B) Compose the policy with a veto that reads the normalized error code, for example `retry_policies.all(retry_policies.http_status([429, 503]), not_billing)`, where `not_billing` returns `False` for `credit_balance_exhausted` and the spend/usage-limit codes.
- C) Remove `ModelRetrySettings` and wrap `Runner.run` in a decorator that retries `RateLimitError`.
- D) Replace `http_status` with `provider_suggested()`, which never retries `429`.

**Q6.** A compliance pipeline's sanctions-check worker calls a vendor API. The vendor's licence expired yesterday and it answers `403`. The worker's tool catches all exceptions and returns `{"matches": []}`, and the report says "Counterparty cleared: no sanctions matches." What is the root fix?

- A) Retry the vendor call with exponential backoff until it succeeds.
- B) Ask the coordinator model to double-check suspiciously empty results.
- C) Log the exception and keep returning an empty list.
- D) Return a distinct error status from the tool (category `permission`, not retryable), make the worker report `failure` with that type, and have the coordinator mark the check NOT COVERED and escalate, never "cleared".

**Q7.** A coordinator must collect results from four specialists, cope with any of them failing, and write one synthesis with coverage notes. Which wiring fits best?

- A) Agents as tools (or code orchestration), so control returns to the coordinator after each specialist, with each specialist returning a structured result.
- B) Handoffs from the coordinator to each specialist in turn.
- C) One agent with all four specialists' instructions concatenated.
- D) Hosted multi-agent with no structured output, reading only the root's final answer.

**Q8.** A background response comes back with each of these terminal states. Which one deserves an automatic, bounded retry (after checking side effects)?

- A) `incomplete` with `incomplete_details.reason == "max_output_tokens"`
- B) `failed` with `error.code == "misalignment_policy_violation"`
- C) `failed` with `error.code == "server_error"`
- D) `cancelled`

**Q9.** An Agents API turn fails with `server_overloaded` after the agent had already sent two of five customer e-mails. What is the right recovery?

- A) Immediately resend the original user message so the session starts over.
- B) Retrieve the session, the turn and its saved items; confirm which e-mails were sent; wait with a limit (honoring `Retry-After`); then, when the session is `idle`, send a follow-up asking it to continue only the unfinished work.
- C) Delete the session and create a new one with the same input.
- D) Treat `idle` as success and close the ticket.

**Q10.** A synthesis agent writes the final report from worker results. Reviewers find a section labelled "complete" although its worker returned `partial_failure`. What is the most reliable fix?

- A) Add "be honest about coverage" to the synthesis agent's instructions.
- B) Use a bigger model for synthesis.
- C) Ask the synthesis agent to rate its own coverage from 1 to 10 at the end.
- D) Compute the coverage map in code from the worker statuses, pass it to the synthesis step as data, and verify the report's per-section coverage claims against it (for example with an output guardrail) before publishing.

In [ ]:
# Fill in "A", "B", "C" or "D" for each question.
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
}

In [ ]:
ANSWER_KEY = {
    "q1": "96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365",
    "q2": "d14b6ae0cbc51f4b715924684653e0d81035a8fe8b79de34ff29eb9db56e2d6b",
    "q3": "2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3",
    "q4": "69a9828043db3a177204403c987facd12c855928421b3aeeb5edce8407e3196c",
    "q5": "fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c",
    "q6": "70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9",
    "q7": "8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1",
    "q8": "89443923f22dc808ba069cfb289117719cbe7d8d36703c74314c6b70fba0371b",
    "q9": "b25dce13077091b19d36fd594efa7fa456f167c4a463522f004183e5d5d2bb5e",
    "q10": "c72adbd855b96e33ffae9308bfc1d5cdfa7fb315301530c3b1f890435529ec04",
}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_KEY, key=lambda q: int(q[1:])):
        given = str(answers.get(qid) or "").strip().upper()
        ok = hashlib.sha256(f"{qid}:{given}".encode()).hexdigest() == ANSWER_KEY[qid]
        correct += ok
        print(f"{qid:>4}: {'✓' if ok else '✗'}  (your answer: {given or '-'})")
    QUIZ_SCORE["correct"] = correct
    print(f"\nQuiz: {correct}/{len(ANSWER_KEY)} = {correct / len(ANSWER_KEY):.0%}")
    return correct


quiz_correct = grade_quiz(answers)

## Part B — Coding exercises (offline)

### Exercise 1 — Classify OpenAI errors (with non-retryable `429`s)

Write `classify_error(exc)` that returns `{"category": ..., "retryable": ...}` for an exception raised by the `openai` client (after its own retries):

| Exception | category | retryable |
|---|---|---|
| `APITimeoutError`, `APIConnectionError` | `"transient"` | `True` |
| `RateLimitError` whose `code` is in `NON_RETRYABLE_429` | `"business"` | `False` |
| any other `RateLimitError` | `"transient"` | `True` |
| `InternalServerError` (any `≥500`) | `"transient"` | `True` |
| `AuthenticationError`, `PermissionDeniedError` | `"permission"` | `False` |
| `BadRequestError`, `UnprocessableEntityError`, `NotFoundError` | `"validation"` | `False` |
| anything else | `"unknown"` | `False` |

In [ ]:
def classify_error(exc: Exception) -> dict:
    # TODO: return {"category": ..., "retryable": ...}
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    try:
        cases = [
            ("timeout", read_timeout, "transient", True),
            ("connection", connect_failure, "transient", True),
            ("429 slow_down", lambda r: error_response(429, "slow_down", "rate_limit_error"), "transient", True),
            ("429 without code", lambda r: error_response(429), "transient", True),
            ("429 credit_balance_exhausted", lambda r: error_response(429, "credit_balance_exhausted"), "business", False),
            ("429 organization_usage_limit_exceeded", lambda r: error_response(429, "organization_usage_limit_exceeded"),
             "business", False),
            ("503 server_is_overloaded", lambda r: error_response(503, "server_is_overloaded"), "transient", True),
            ("500", lambda r: error_response(500), "transient", True),
            ("400", lambda r: error_response(400, None, "invalid_request_error", "tools"), "validation", False),
            ("404", lambda r: error_response(404), "validation", False),
            ("401", lambda r: error_response(401), "permission", False),
            ("403", lambda r: error_response(403), "permission", False),
            ("409", lambda r: error_response(409), "unknown", False),
        ]
        for label, handler, category, retryable in cases:
            got = classify_error(raise_from(handler))
            assert isinstance(got, dict), "return a dict with 'category' and 'retryable'"
            assert got.get("category") == category, f"{label}: expected category {category!r}, got {got.get('category')!r}" + (
                " (a 429 is not always transient: read exc.code)" if label.startswith("429 credit") or "usage" in label else "")
            assert got.get("retryable") is retryable, f"{label}: expected retryable={retryable}, got {got.get('retryable')!r}"
        record(1, True)
    except NotImplementedError:
        record(1, False, "Implement classify_error first.")
    except Exception as e:  # never crash the notebook
        record(1, False, f"{type(e).__name__}: {e}")


check_ex1()

### Exercise 2 — A structured `failure_error_function`

Write `structured_tool_error(ctx, error) -> str` for the Agents SDK. It must return a **JSON string** with exactly these keys: `status` (always `"error"`), `category`, `retryable`, `message`, `suggestion`.

- `ToolFailure` → use its `category`, `retryable`, `str(error)` and `suggestion`.
- `TimeoutError` → `category` `"transient"`, `retryable` `True`, a message of your own, and a non-empty suggestion.
- anything else → `category` `"unknown"`, `retryable` `False`, a message that names the exception **type** and the tool (`ctx.tool_name`), and a non-empty suggestion. Never include the exception's own text (it may contain hostnames or secrets).

The checker plugs your function into a real `@function_tool` and runs it with a scripted model.

In [ ]:
def structured_tool_error(ctx: RunContextWrapper, error: Exception) -> str:
    # TODO
    raise NotImplementedError

In [ ]:
async def check_ex2() -> None:
    try:
        try:                                   # the runner would wrap NotImplementedError; probe it directly first
            structured_tool_error(None, TimeoutError())
        except NotImplementedError:
            raise
        except Exception:
            pass
        failures = {
            "validation": ToolFailure("validation", "Unknown topic 'sculpture'.", False, "Use music or literature."),
            "timeout": TimeoutError("504 from search-internal.corp:8443"),
            "unexpected": KeyError("password=hunter2 at db-prod-7.corp"),
        }

        async def output_for(name: str) -> str:
            @function_tool(failure_error_function=structured_tool_error)
            async def search_archive(query: str) -> str:
                """Search the archive.

                Args:
                    query: Search terms.
                """
                raise failures[name]

            model = ScriptedModel([[function_call("search_archive", {"query": "q"}, call_id="c1")],
                                   [assistant_message("done")]])
            result = await Runner.run(Agent(name="w", model=model, tools=[search_archive]), "go", run_config=OFFLINE)
            return next(i.output for i in result.new_items if i.type == "tool_call_output_item")

        keys = {"status", "category", "retryable", "message", "suggestion"}
        v = json.loads(await output_for("validation"))
        assert set(v) == keys, f"expected exactly the keys {sorted(keys)}, got {sorted(v)}"
        assert v["status"] == "error" and v["category"] == "validation" and v["retryable"] is False, \
            f"ToolFailure: use its category/retryable, got {v}"
        assert v["message"] == "Unknown topic 'sculpture'." and v["suggestion"] == "Use music or literature.", \
            "ToolFailure: message is str(error), suggestion is error.suggestion"
        t = json.loads(await output_for("timeout"))
        assert (t["category"], t["retryable"]) == ("transient", True), f"TimeoutError -> transient/True, got {t}"
        assert t["suggestion"], "give the model a suggestion for timeouts"
        assert "corp" not in json.dumps(t), "the timeout message leaks the internal hostname: write your own message"
        u_raw = await output_for("unexpected")
        u = json.loads(u_raw)
        assert (u["category"], u["retryable"]) == ("unknown", False), f"unexpected errors -> unknown/False, got {u}"
        assert "hunter2" not in u_raw and "db-prod" not in u_raw, "never put the exception text into model-visible output"
        assert "KeyError" in u["message"] and "search_archive" in u["message"], \
            "name the exception type and the tool (ctx.tool_name) in the message"
        assert u["suggestion"], "give the model a suggestion for unexpected errors too"
        record(2, True)
    except NotImplementedError:
        record(2, False, "Implement structured_tool_error first.")
    except json.JSONDecodeError:
        record(2, False, "Return a JSON string (json.dumps), not plain text.")
    except Exception as e:
        record(2, False, f"{type(e).__name__}: {e}")


await check_ex2()

### Exercise 3 — A bounded retry helper with a deadline

Write `async def retry_with_deadline(op, *, attempts=3, deadline_s=10.0, base_delay=0.1)`:

- `await op()` and return its result.
- Retry **only** transient errors: `TimeoutError` and `ConnectionError`. Any other exception propagates immediately (no retry).
- Stop after `attempts` total calls, **or** when the next attempt would start after `deadline_s` seconds since the first call, whichever comes first, and re-raise the last transient error.
- Wait between attempts with exponential backoff (`base_delay * 2 ** (attempt - 1)` is fine), but never sleep past the deadline.

In [ ]:
async def retry_with_deadline(op, *, attempts: int = 3, deadline_s: float = 10.0, base_delay: float = 0.1):
    # TODO
    raise NotImplementedError

In [ ]:
def _flaky(errors: list[BaseException], delay: float = 0.0):
    state = {"calls": 0}

    async def op():
        state["calls"] += 1
        if delay:
            await asyncio.sleep(delay)
        if errors:
            raise errors.pop(0)
        return "ok"
    return op, state


async def _bounded(coro, limit: float = 3.0):
    """Run one call of the learner's helper, but never let it hang the notebook."""
    task = asyncio.ensure_future(coro)
    done, _ = await asyncio.wait({task}, timeout=limit)
    if not done:
        task.cancel()
        raise AssertionError(f"retry_with_deadline did not finish within {limit} s: respect attempts AND deadline_s")
    return task.result()


async def check_ex3() -> None:
    try:
        op, s = _flaky([TimeoutError()])
        assert await _bounded(retry_with_deadline(op, attempts=3, base_delay=0.001)) == "ok" and s["calls"] == 2, \
            f"one transient error then success: expected 2 calls, got {s['calls']}"
        op, s = _flaky([ConnectionError()] * 10)
        try:
            await _bounded(retry_with_deadline(op, attempts=3, base_delay=0.001))
            raise AssertionError("a persistent transient error must be re-raised after the last attempt")
        except ConnectionError:
            pass
        assert s["calls"] == 3, f"attempts=3 means exactly 3 calls, got {s['calls']}"
        op, s = _flaky([PermissionError("403")])
        try:
            await _bounded(retry_with_deadline(op, attempts=3, base_delay=0.001))
            raise AssertionError("PermissionError must propagate")
        except PermissionError:
            pass
        assert s["calls"] == 1, f"non-transient errors must not be retried (got {s['calls']} calls)"
        op, s = _flaky([ValueError("bad input")])
        try:
            await _bounded(retry_with_deadline(op, attempts=3, base_delay=0.001))
            raise AssertionError("ValueError must propagate, not be swallowed")
        except ValueError:
            pass
        assert s["calls"] == 1, "validation-type errors (ValueError) must not be retried"
        op, s = _flaky([TimeoutError()] * 100, delay=0.05)
        start = time.monotonic()
        try:
            await _bounded(retry_with_deadline(op, attempts=100, deadline_s=0.2, base_delay=0.01))
            raise AssertionError("expected the last TimeoutError after the deadline")
        except TimeoutError:
            pass
        elapsed = time.monotonic() - start
        assert elapsed < 0.6, f"the deadline (0.2 s) was not respected: took {elapsed:.2f} s"
        assert s["calls"] <= 5, f"deadline 0.2 s with 0.05 s calls should stop after a few calls, got {s['calls']}"
        op, s = _flaky([TimeoutError()] * 5)
        start = time.monotonic()
        try:
            await _bounded(retry_with_deadline(op, attempts=3, deadline_s=10, base_delay=0.05))
        except TimeoutError:
            pass
        assert time.monotonic() - start >= 0.05, "wait (back off) between attempts"
        record(3, True)
    except NotImplementedError:
        record(3, False, "Implement retry_with_deadline first.")
    except Exception as e:
        record(3, False, f"{type(e).__name__}: {e}")


await check_ex3()

### Exercise 4 — Build a `WorkerResult` that separates valid-empty from failure

Write `build_worker_result(topic, tool_outputs)`. `tool_outputs` is a list of `(query, output_json)` pairs, one per search the worker ran. Each output is either `{"status": "ok", "results": [{"title": ..., "source": ...}, ...]}` (results may be empty) or `{"status": "error", "category": ..., "retryable": ..., "message": ...}`.

- `findings`: every result from every `ok` output, in order.
- `status`: `"success"` if no output is an error (**empty results are still success**); `"partial_failure"` if at least one `ok` and at least one error; `"failure"` if all are errors.
- `failure_type`: `None` on success; otherwise map each error's category (`transient` → `"timeout"`, `permission` → `"permission"`, `validation` → `"validation"`, anything else → `"unknown"`) and return the **most actionable** one, whatever its position: `permission` > `validation` > `unknown` > `timeout`. (A timeout that happens to come first must not hide a permission failure that the coordinator has to escalate.)
- `retryable`: `True` only if there is at least one error and **every** error is retryable.
- `attempted_query`: the queries joined with `"; "`.
- `alternative_approaches`: `["Retry with a narrower query"]` if any error is transient, else `[]`.
- `coverage_impact`: `None` on success; otherwise `"Not covered: "` followed by the failed queries joined with `", "`.

In [ ]:
def build_worker_result(topic: str, tool_outputs: list[tuple[str, str]]) -> WorkerResult:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def ok(*titles):
        return json.dumps({"status": "ok", "results": [{"title": t, "source": f"archive#{i}"} for i, t in enumerate(titles)]})

    def err(category, retryable):
        return json.dumps({"status": "error", "category": category, "retryable": retryable, "message": "x"})

    try:
        r = build_worker_result("literature", [("AI novels", ok())])
        assert isinstance(r, WorkerResult), "return a WorkerResult"
        assert r.status == "success" and r.failure_type is None and r.coverage_impact is None, \
            f"a search that ran and found nothing is SUCCESS (valid empty), got status={r.status}"
        assert r.findings == [] and r.retryable is False, "valid empty: no findings, not retryable"
        r = build_worker_result("music", [("AI music generation", ok("Gen report")),
                                          ("AI vocal synthesis", err("transient", True))])
        assert r.status == "partial_failure", f"one ok + one error -> partial_failure, got {r.status}"
        assert [f.title for f in r.findings] == ["Gen report"], "keep the findings from the successful search"
        assert r.failure_type == "timeout" and r.retryable is True, "transient -> failure_type 'timeout', retryable"
        assert r.attempted_query == "AI music generation; AI vocal synthesis", "join queries with '; '"
        assert r.alternative_approaches == ["Retry with a narrower query"], "suggest a narrower query for transient errors"
        assert r.coverage_impact == "Not covered: AI vocal synthesis", f"coverage_impact wrong: {r.coverage_impact!r}"
        r = build_worker_result("visual_art", [("licensed sales", err("permission", False)),
                                               ("archive", err("transient", True))])
        assert r.status == "failure", "all searches failed -> failure"
        assert r.failure_type == "permission", "permission outranks a timeout"
        assert r.retryable is False, "retryable only if EVERY error is retryable"
        assert r.coverage_impact == "Not covered: licensed sales, archive", f"coverage_impact wrong: {r.coverage_impact!r}"
        r = build_worker_result("design", [("archive", err("transient", True)), ("licensed sales", err("permission", False))])
        assert r.failure_type == "permission", \
            "the most actionable error wins whatever its position (a timeout first must not hide a permission failure)"
        assert r.retryable is False, "retryable only if EVERY error is retryable"
        r = build_worker_result("games", [("a", err("transient", True)), ("b", err("validation", False)),
                                          ("c", err("business", False))])
        assert r.failure_type == "validation", "precedence: permission > validation > unknown > timeout"
        r = build_worker_result("film", [("q", err("business", False))])
        assert r.failure_type == "unknown" and r.alternative_approaches == [], "unmapped categories -> 'unknown', no alternatives"
        record(4, True)
    except NotImplementedError:
        record(4, False, "Implement build_worker_result first.")
    except Exception as e:
        record(4, False, f"{type(e).__name__}: {e}")


check_ex4()

### Exercise 5 — The coordinator's decision function

Write `coordinator_decision(r, already_redispatched=False)` that returns one of these strings, checking the rules **in this order**:

1. `status == "success"` → `"use"` if there are findings, else `"use_empty"`
2. `failure_type == "permission"` → `"escalate"`
3. `failure_type == "policy_blocked"` → `"escalate"`
4. `failure_type == "rate_limited"` and not retryable → `"stop_workflow"`
5. retryable, at least one alternative approach, and not already re-dispatched → `"redispatch"`
6. otherwise `"accept_partial"` if there are findings, else `"report_gap"`

In [ ]:
def coordinator_decision(r: WorkerResult, already_redispatched: bool = False) -> str:
    # TODO
    raise NotImplementedError

In [ ]:
def _wr(status, failure_type=None, retryable=False, findings=0, alternatives=()):
    return WorkerResult(topic="t", status=status, failure_type=failure_type, retryable=retryable, attempted_query="q",
                        findings=[Finding(title=f"f{i}", source="s") for i in range(findings)],
                        alternative_approaches=list(alternatives), coverage_impact=None if status == "success" else "gap")


def check_ex5() -> None:
    try:
        cases = [
            (_wr("success", findings=2), False, "use"),
            (_wr("success"), False, "use_empty"),
            (_wr("partial_failure", "permission", False, 1), False, "escalate"),
            (_wr("failure", "policy_blocked"), False, "escalate"),
            (_wr("failure", "rate_limited", False), False, "stop_workflow"),
            (_wr("failure", "rate_limited", True, 0, ["wait and retry later"]), False, "redispatch"),
            (_wr("partial_failure", "timeout", True, 1, ["narrower query"]), False, "redispatch"),
            (_wr("partial_failure", "timeout", True, 1, ["narrower query"]), True, "accept_partial"),
            (_wr("failure", "timeout", True, 0, []), False, "report_gap"),
            (_wr("failure", "validation", False, 0, ["fix input"]), False, "report_gap"),
            (_wr("partial_failure", "max_turns", False, 3, ["split"]), False, "accept_partial"),
        ]
        hints = {"use_empty": "an empty SUCCESS is not a gap",
                 "escalate": "permission and policy blocks are never retried",
                 "stop_workflow": "a non-retryable rate_limited result means billing/quota: stop new work",
                 "redispatch": "re-dispatch once when retryable and an alternative exists",
                 "accept_partial": "re-dispatch at most once; keep the partial findings",
                 "report_gap": "nothing usable and nothing left to try"}
        for r, redispatched, want in cases:
            got = coordinator_decision(r, redispatched)
            assert got == want, (f"status={r.status} failure_type={r.failure_type} retryable={r.retryable} "
                                 f"findings={len(r.findings)} redispatched={redispatched}: expected {want!r}, got {got!r} "
                                 f"({hints.get(want, '')})")
        record(5, True)
    except NotImplementedError:
        record(5, False, "Implement coordinator_decision first.")
    except Exception as e:
        record(5, False, f"{type(e).__name__}: {e}")


check_ex5()

### Exercise 6 — A coverage-annotated report renderer

Write `render_coverage_report(title, results)` that returns a Markdown string:

- First line: `## <title>`
- One section per result, in order, with a heading `### <Topic> (<LABEL>)`, where `<Topic>` is `topic` with `_` replaced by spaces and title-cased (`visual_art` → `Visual Art`), and `<LABEL>` is `FULL COVERAGE`, `PARTIAL COVERAGE` or `NOT COVERED` for `success`, `partial_failure`, `failure`. If `failure_type` is set, append `: <failure_type>` (e.g. `PARTIAL COVERAGE: timeout`). A success without findings gets `; searched, no matching records` appended.
- Under each heading: one `- <title> (<source>)` line per finding, or `- No findings.`; then `> Coverage note: <coverage_impact>` if it is set.
- Last line: `**Coverage:** <n complete>/<n total> sections complete`, followed by `; gaps: <topics joined with ", ">` if any section is not `success`.

In [ ]:
def render_coverage_report(title: str, results: list[WorkerResult]) -> str:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    try:
        results = [
            WorkerResult(topic="visual_art", status="success", failure_type=None, retryable=False, attempted_query="q",
                         findings=[Finding(title="AI art report", source="archive#12"),
                                   Finding(title="Gallery adoption survey", source="archive#31")],
                         alternative_approaches=[], coverage_impact=None),
            WorkerResult(topic="music", status="partial_failure", failure_type="timeout", retryable=True, attempted_query="q",
                         findings=[Finding(title="AI songwriting tools survey", source="archive#7")],
                         alternative_approaches=["narrower"], coverage_impact="Not covered: AI vocal synthesis."),
            WorkerResult(topic="literature", status="success", failure_type=None, retryable=False, attempted_query="q",
                         findings=[], alternative_approaches=[], coverage_impact=None),
            WorkerResult(topic="film", status="failure", failure_type="permission", retryable=False, attempted_query="q",
                         findings=[], alternative_approaches=[], coverage_impact="Licensed database not accessible."),
        ]
        text = render_coverage_report("AI impact on creative industries", results)
        assert isinstance(text, str), "return a string"
        lines = [l.rstrip() for l in text.strip().splitlines()]
        assert lines[0] == "## AI impact on creative industries", "first line must be '## <title>'"
        for heading in ["### Visual Art (FULL COVERAGE)", "### Music (PARTIAL COVERAGE: timeout)",
                        "### Literature (FULL COVERAGE; searched, no matching records)",
                        "### Film (NOT COVERED: permission)"]:
            assert heading in lines, f"missing heading line: {heading!r}"
        sections, current = {}, None                    # heading -> the '- ' lines under it
        for line in lines[1:]:
            if line.startswith("### "):
                current = line.split(" (")[0][4:]
                sections[current] = []
            elif current and line.startswith("- "):
                sections[current].append(line)
        expected = {"Visual Art": ["- AI art report (archive#12)", "- Gallery adoption survey (archive#31)"],
                    "Music": ["- AI songwriting tools survey (archive#7)"],
                    "Literature": ["- No findings."], "Film": ["- No findings."]}
        for topic, bullets in expected.items():
            assert sections.get(topic) == bullets, (
                f"{topic}: expected {bullets} under its heading, got {sections.get(topic)} "
                "(one '- <title> (<source>)' line per finding, partial sections included; '- No findings.' if none)")
        assert "> Coverage note: Not covered: AI vocal synthesis." in lines, "add the coverage note for music"
        assert "> Coverage note: Licensed database not accessible." in lines, "add the coverage note for film"
        assert sum(l.startswith("> Coverage note:") for l in lines) == 2, "only sections with coverage_impact get a note"
        order = [l for l in lines if l.startswith("### ")]
        assert [o.split(" (")[0] for o in order] == ["### Visual Art", "### Music", "### Literature", "### Film"], \
            "keep the results' order"
        assert lines[-1] == "**Coverage:** 2/4 sections complete; gaps: music, film", f"wrong last line: {lines[-1]!r}"
        full = render_coverage_report("T", [results[0]]).strip().splitlines()[-1].rstrip()
        assert full == "**Coverage:** 1/1 sections complete", f"no gaps -> no '; gaps' suffix, got {full!r}"
        record(6, True)
    except NotImplementedError:
        record(6, False, "Implement render_coverage_report first.")
    except Exception as e:
        record(6, False, f"{type(e).__name__}: {e}")


check_ex6()

## Part C — Architecture scenario

A private-equity firm wants an **automated due-diligence pipeline** on the OpenAI platform. For each target company, a coordinator dispatches five worker agents (financials, litigation, sanctions and PEP screening, news sentiment, patents). Constraints:

- 200 targets must be processed overnight; the project is on a mid usage tier and has a monthly **project spend limit**.
- The patents worker uses a **paywalled** patent database whose subscription is managed by another team and sometimes lapses.
- The sanctions worker must never report "clear" unless the screening actually ran.
- Some workers call tools with **side effects** (they open tickets in the firm's case-management system).
- Analysts read the final memo; compliance audits every step.

Describe your design for failure handling: how you classify and route errors, where retries live and how they are bounded, what each worker returns, how the coordinator decides, how the memo shows coverage, how you protect against duplicate side effects, and how you debug a bad night. Name at least two trade-offs.

*Your answer:*

-
-
-

<details><summary>Rubric: what a strong answer contains</summary>

- **Classification by code, not status.** `429 slow_down` / plain rate limits and `503 server_is_overloaded` are transient; `project_spend_limit_exceeded` and `credit_balance_exhausted` are business errors that **stop new work for the whole run** and alert an owner; `401`/`403` and the paywall's `403` are permission; `400` validation.
- **Throughput without `slow_down`.** A concurrency cap (semaphore) over targets × workers and a gradual ramp, rather than bigger retry counts; possibly Batch or Flex for non-urgent parts (Chapter 07) with its own failure statuses.
- **One owning retry layer per failure type, with a deadline.** Client retries counted (or `max_retries=0` where the app owns them); `ModelRetrySettings` with a policy that vetoes billing codes; tool-level single retry for idempotent reads; worker retries a transient tool error once with a changed query; coordinator re-dispatches at most once; per-worker and per-target deadlines (`asyncio.timeout`), circuit breaker for the patent DB during outages.
- **Errors as data.** Custom `failure_error_function` on every tool and on every `as_tool` (no generic default, no leaked internals); async tools with `timeout=`.
- **Structured worker results.** `WorkerResult` as `output_type` (status, failure type, retryable, attempted query, partial findings, alternatives, coverage impact); a wrapper that converts `MaxTurnsExceeded` (salvaging `run_data.new_items`), guardrail tripwires and `openai.APIError` into results; `gather`/wrappers, not `TaskGroup`.
- **Sanctions rule.** "Clear" only when the screening ran and returned no matches (valid empty); any failure there marks the memo **NOT COVERED** and blocks sign-off, never "cleared".
- **Paywall.** Permission failure → no retry, section annotated, access escalated to the owning team (Chapter 09); optionally reroute to a secondary source with provenance labelled.
- **Coverage in the memo.** Computed in code from worker statuses, rendered per section (FULL / PARTIAL / NOT COVERED with notes); an output guardrail checks the model-written memo's coverage claims; too many gaps → the memo is marked partial or held.
- **Side effects.** Idempotency keys on ticket creation; check existing tickets before retrying; never replay a failed turn or background request blindly; Agents API recovery procedure if a managed runtime is used.
- **Debugging and audit.** One `trace(...)` per target with `group_id` = target ID; `custom_span` per worker with `set_error`; request IDs logged; `trace_include_sensitive_data=False` if traces must not hold personal data (note: no tracing under ZDR); failed cases become eval cases (Chapter 06).
- **Trade-offs named**, for example: more retries (recovery) vs cost and `slow_down` risk; strict "NOT COVERED blocks sign-off" (safety) vs analyst throughput; a secondary patent source (coverage) vs provenance and licence risk; Batch (cost) vs latency.
</details>

## Scorecard

In [ ]:
quiz_correct, quiz_total = QUIZ_SCORE["correct"], QUIZ_SCORE["total"]
ex_total = 6
ex_passed = sum(EX_RESULTS.get(n, False) for n in range(1, ex_total + 1))
overall = (quiz_correct + ex_passed) / (quiz_total + ex_total)
print(f"Quiz:      {quiz_correct}/{quiz_total}")
print(f"Exercises: {ex_passed}/{ex_total}  " + " ".join(f"Ex{n}:{'✓' if EX_RESULTS.get(n) else '✗'}" for n in range(1, ex_total + 1)))
print(f"Overall:   {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")
print("Part C is self-assessed against the rubric.")

Next chapter: [Chapter 11 — Context Management in Production Systems](../11-context-management/README.md)

---

## Solutions (spoilers)

<details><summary>Quiz answers with explanations</summary>

- **Q1 — B.** The same `429` status covers throttling and billing/quota. The error-codes guide gives `slow_down` a "follow `Retry-After`, ramp gradually" fix and the spend limit a "raise or remove the limit" fix; the rate-limit guide says not to retry errors that require you to act. A burns requests on a limit that will not lift; C misclassifies; D doesn't bypass limits.
- **Q2 — D.** `max_retries=2` is the default, and the retry rule is status-based (`408`, `409`, `429`, `≥500`, unless `x-should-retry` says otherwise). Nested layers multiply, so one layer should own retries. A, B and C are not how the platform behaves.
- **Q3 — A.** `as_tool` has its own `failure_error_function`, whose default is the same generic "Please try again" message. A structured error tells the coordinator whether to retry, reroute or annotate. B kills the healthy parts of the run; C gives the loop more room; D takes the coordinator out of the loop.
- **Q4 — C.** The client honors `Retry-After` only up to 120 s; above that it does not retry at all, so you see one request. The failure is still transient, but a 5-minute wait must fit your deadline; otherwise report the gap. A is wrong (the client retries every `≥500`) and a tight loop ignores the server's request; B is wrong (the default is 2); D ignores an explicit instruction from the server.
- **Q5 — B.** `http_status([429])` retries every `429`, including billing. A veto composed with `retry_policies.all(...)` keeps transient retries and stops billing retries. A still retries billing; C adds a second retry layer on top of the client's, and it is just as status-based; D is wrong: with the OpenAI model, provider advice suggests retrying `429`s unless the server sends `x-should-retry: false`.
- **Q6 — D.** This is silent suppression with compliance consequences. "No matches" is a valid empty result only when the screening ran. A loops on a non-retryable error; B asks a model to recover information that is no longer in the data; C keeps the false "cleared".
- **Q7 — A.** Agents-as-tools and code orchestration return control to the coordinator, which can handle each failure and synthesize. B hands control away; C loses separation and makes failures harder to isolate; D gives you the least control over recovery.
- **Q8 — C.** `server_error` is transient. `incomplete` with `max_output_tokens` repeats with the same settings (A); `misalignment_policy_violation` must never be retried (B); `cancelled` is usually intentional (D).
- **Q9 — B.** This is the Agents API recovery procedure: check the outcome, check completed actions, wait with a limit, continue only unfinished work. A and C risk duplicate e-mails; D confuses `idle` ("ready for input") with success.
- **Q10 — D.** Models smooth over gaps; a code-computed map plus a deterministic check cannot. A and C ask the model to grade itself; B doesn't change the incentive.
</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def classify_error(exc: Exception) -> dict:
    def out(category: str, retryable: bool) -> dict:
        return {"category": category, "retryable": retryable}

    if isinstance(exc, openai.APIConnectionError):            # includes APITimeoutError
        return out("transient", True)
    if isinstance(exc, openai.RateLimitError):
        return out("business", False) if exc.code in NON_RETRYABLE_429 else out("transient", True)
    if isinstance(exc, openai.InternalServerError):
        return out("transient", True)
    if isinstance(exc, (openai.AuthenticationError, openai.PermissionDeniedError)):
        return out("permission", False)
    if isinstance(exc, (openai.BadRequestError, openai.UnprocessableEntityError, openai.NotFoundError)):
        return out("validation", False)
    return out("unknown", False)
```
</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def structured_tool_error(ctx: RunContextWrapper, error: Exception) -> str:
    if isinstance(error, ToolFailure):
        payload = {"category": error.category, "retryable": error.retryable,
                   "message": str(error), "suggestion": error.suggestion}
    elif isinstance(error, TimeoutError):
        payload = {"category": "transient", "retryable": True, "message": "The upstream search timed out.",
                   "suggestion": "Retry once with a narrower query, then report partial coverage."}
    else:
        tool = getattr(ctx, "tool_name", "tool")
        payload = {"category": "unknown", "retryable": False,
                   "message": f"{type(error).__name__} in {tool}",
                   "suggestion": "Do not retry. Report this source as not covered."}
    return json.dumps({"status": "error", **payload})
```
</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
async def retry_with_deadline(op, *, attempts: int = 3, deadline_s: float = 10.0, base_delay: float = 0.1):
    start = time.monotonic()
    for attempt in range(1, attempts + 1):
        try:
            return await op()
        except (TimeoutError, ConnectionError):
            remaining = deadline_s - (time.monotonic() - start)
            delay = base_delay * 2 ** (attempt - 1)
            if attempt == attempts or remaining <= delay:
                raise
            await asyncio.sleep(delay)
```
</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
FAILURE_TYPES = {"transient": "timeout", "permission": "permission", "validation": "validation"}
PRECEDENCE = ["permission", "validation", "unknown", "timeout"]          # most actionable first


def build_worker_result(topic: str, tool_outputs: list[tuple[str, str]]) -> WorkerResult:
    parsed = [(query, json.loads(raw)) for query, raw in tool_outputs]
    findings = [Finding(**hit) for _, out in parsed if out["status"] == "ok" for hit in out["results"]]
    errors = [(query, out) for query, out in parsed if out["status"] == "error"]
    n_ok = len(parsed) - len(errors)
    status = "success" if not errors else ("partial_failure" if n_ok else "failure")
    return WorkerResult(
        topic=topic,
        status=status,
        failure_type=min((FAILURE_TYPES.get(out.get("category"), "unknown") for _, out in errors),
                         key=PRECEDENCE.index) if errors else None,
        retryable=bool(errors) and all(out.get("retryable") for _, out in errors),
        attempted_query="; ".join(query for query, _ in parsed),
        findings=findings,
        alternative_approaches=["Retry with a narrower query"]
        if any(out.get("category") == "transient" for _, out in errors) else [],
        coverage_impact="Not covered: " + ", ".join(query for query, _ in errors) if errors else None,
    )
```
</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def coordinator_decision(r: WorkerResult, already_redispatched: bool = False) -> str:
    if r.status == "success":
        return "use" if r.findings else "use_empty"
    if r.failure_type in ("permission", "policy_blocked"):
        return "escalate"
    if r.failure_type == "rate_limited" and not r.retryable:
        return "stop_workflow"
    if r.retryable and r.alternative_approaches and not already_redispatched:
        return "redispatch"
    return "accept_partial" if r.findings else "report_gap"
```
</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
LABELS = {"success": "FULL COVERAGE", "partial_failure": "PARTIAL COVERAGE", "failure": "NOT COVERED"}


def render_coverage_report(title: str, results: list[WorkerResult]) -> str:
    lines = [f"## {title}", ""]
    for r in results:
        label = LABELS[r.status] + (f": {r.failure_type}" if r.failure_type else "")
        if r.status == "success" and not r.findings:
            label += "; searched, no matching records"
        lines.append(f"### {r.topic.replace('_', ' ').title()} ({label})")
        lines += [f"- {f.title} ({f.source})" for f in r.findings] or ["- No findings."]
        if r.coverage_impact:
            lines.append(f"> Coverage note: {r.coverage_impact}")
        lines.append("")
    gaps = [r.topic for r in results if r.status != "success"]
    lines.append(f"**Coverage:** {len(results) - len(gaps)}/{len(results)} sections complete"
                 + (f"; gaps: {', '.join(gaps)}" if gaps else ""))
    return "\n".join(lines)
```
</details>